# Reactor Yield: Physics + Machine Learning
## A physics guided surrogate for a nonisothermal plug flow reactor

**Author:** Ashish Sharma, Team Outliers · Fugacity 2026 ML Challenge

Predict the exit yield of an intermediate product using a coupled A → B → C reactor model
and a shrunk ExtraTrees residual correction. The workflow compares physical models with five, six,
and seven parameters against a pure ML baseline using repeated cross validation.

**Start here:** install `requirements.txt`, select that Python environment, and run from the
repository root. Supply the original challenge CSVs in `data/raw/`; the input schema is in
[`data/README.md`](data/README.md). Run `python -m scripts.demo` for an illustrative simulation
that does not require the challenge data.

### Evidence and reproducibility
The supplied notebook had no saved outputs or input datasets. This maintained version separates
implemented methods from historical claims; the original notebook and presentation remain in
`archive/` and `reports/`. Historical metrics are documented in [`docs/RESULTS.md`](docs/RESULTS.md).
No results below are considered reproduced until this notebook is executed on the original data.

### Modeling questions
* Does encoding the sequential reaction improve over a tree baseline on this small dataset?
* What changes when the energy balance includes one or both reaction heat terms?
* Does a small residual correction improve held out prediction?
* How do grid resolution and the evaluation metric affect the conclusions?


## 0. Configuration

In [ ]:
import json
import logging
import time
from dataclasses import asdict
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kurtosis, pearsonr
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import KFold
from reactor_model import Config, simulate, fit, features, prepare_conditions, predict_yield
from reactor_model import VARIANTS, rmse, medae, trimmed_rmse, hurdle_predict

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
log = logging.getLogger("reactor")
RNG = 42
CFG = Config()  # Defaults: team Outliers; inputs under data/raw/; outputs under artifacts/
Path(CFG.out_dir).mkdir(parents=True, exist_ok=True)

missing = [path for path in (CFG.train_path, CFG.test_path) if not Path(path).is_file()]
if missing:
    raise FileNotFoundError(
        f"Missing challenge inputs: {missing}. See data/README.md; "
        "the dataset-free example is: python -m scripts.demo"
    )
train, test = pd.read_csv(CFG.train_path), pd.read_csv(CFG.test_path)
D, D_te = prepare_conditions(train), prepare_conditions(test)
y = train[CFG.target].to_numpy(dtype=float)
if not np.isfinite(y).all() or ((y < 0) | (y > 100)).any():
    raise ValueError("Training yield must be finite and between 0 and 100 percent")
assert len(test) == 50, "the challenge submission requires exactly 50 test rows"
Q, C0, Ti, L, Tj, tau = (D[name] for name in ("Q", "C0", "Ti", "L", "Tj", "tau"))
sub = lambda d, i: {k: v[i] for k, v in d.items()}
log.info("train=%s  test=%s", train.shape, test.shape)


## 1. Exploratory checks
Inspect residence time groups within temperature bands, concentration association, and
labels near zero. These plots help formulate hypotheses; a marginal correlation does not
identify reaction order, and zero labels alone do not establish their physical cause.


In [ ]:
T_mix = (Ti + Tj) / 2
probe = train.assign(tau=tau, T_mix=T_mix)
print("Mean yield by L/Q quartile within temperature bands")
for lo, hi in [(350, 410), (410, 440), (440, 470), (470, 520)]:
    band = probe[(probe.T_mix >= lo) & (probe.T_mix < hi)]
    if len(band) < 8:
        continue
    q = pd.qcut(band.tau, 4, duplicates="drop")
    print(f"T_mix [{lo},{hi}), n={len(band)}: "
          f"{band.groupby(q, observed=True)[CFG.target].mean().round(1).to_dict()}")
print("Concentration Pearson correlation:", np.corrcoef(C0, y)[0, 1])
print("Concentration Spearman correlation:", train.concentration_mol_L.corr(train[CFG.target], method="spearman"))
n_zero = int((y < 1e-2).sum())
print(f"Near-zero yields: {n_zero}/{len(y)}; investigate before deciding how to treat them")


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
ax[0].hist(y, bins=30, color="#2b6cb0", edgecolor="white")
ax[0].set_title("Observed target distribution"); ax[0].set_xlabel("overall_yield")

s = ax[1].scatter(tau, T_mix, c=y, cmap="viridis", s=30)
ax[1].set_xlabel(r"residence time $\tau=L/Q$"); ax[1].set_ylabel(r"$T_{mix}$ (K)")
ax[1].set_title("Yield over operating conditions"); plt.colorbar(s, ax=ax[1], label="yield")

for lo, hi, c in [(350, 410, "#2b6cb0"), (410, 440, "#dd6b20"), (440, 520, "#c53030")]:
    m = (T_mix >= lo) & (T_mix < hi)
    ax[2].scatter(tau[m], y[m], s=26, alpha=.75, color=c, label=f"T_mix [{lo},{hi})")
ax[2].set_xlabel(r"$\tau=L/Q$"); ax[2].set_ylabel("overall_yield")
ax[2].set_title("Yield versus L/Q by temperature"); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()


## 2. Governing equations and numerical model

Let $s=z/L$ denote normalized axial position and $\tau=L/Q$ denote the notebook's residence time
group. Because the reactor's cross section is not supplied here, $L/Q$ is not a dimensional residence
time: fitted kinetic and heat transfer coefficients absorb that scaling.

$$\frac{dy_A}{ds}=-\tau k_1 y_A,\qquad
\frac{dy_B}{ds}=\tau(k_1y_A-k_2y_B)$$

$$\frac{dT}{ds}=h\tau(T_j-T)+\beta_1 C_{A0}\tau k_1y_A+\beta_2 C_{A0}\tau k_2y_B$$

$$k_i(T)=\exp\left[\ln k_{i,ref}-\frac{E_i}{R}\left(\frac1T-\frac1{T_{ref}}\right)\right]$$

The seven effective parameters are $\ln k_{1,ref}$, $\ln k_{2,ref}$, $E_1/R$, $E_2/R$, $h$,
$\beta_1$, and $\beta_2$. The implementation assumes first order sequential reactions; it does
not establish mechanistic identifiability or include a direct A → C route.

The vectorized exponential midpoint scheme is implemented in `reactor_model.py`. `expm1` avoids
cancellation in the small rate limit. Some frozen linear terms are integrated exactly, but the
coupled update is approximate: neither exactness of the whole scheme nor unconditional stability
of the nonlinear thermal system is claimed. Grid convergence must be checked in the relevant regime.


In [ ]:
# The notebook, demo, and tests share this implementation.
# Inspect reactor_model.py for the exponential updates and robust fitting utilities.
print("Shared physics solver loaded:", simulate.__module__)


## 3. Numerical diagnostics
Compare the solver with jacket exchange only to an independent integrating factor calculation
evaluated with numerical quadrature. Inspect axial convergence against a finer grid. The automated
test suite also checks cases at constant temperature against a closed form solution for sequential
reactions. These checks cover selected conditions, not every possible stiff thermal regime.


In [ ]:
# --- check 1: agreement with the integrating-factor reference of the beta=0 model ---
_S = np.linspace(0.0, 1.0, 201); _dS = _S[1] - _S[0]

def simulate_analytic(p5, d, cfg: Config = CFG):
    """Integrating-factor reference with numerical quadrature; beta1 = beta2 = 0 only."""
    ln_k1, ln_k2, E1_R, E2_R, h = p5
    t, Ti_, Tj_ = d["tau"], d["Ti"], d["Tj"]
    T = Tj_[:, None] + (Ti_ - Tj_)[:, None] * np.exp(-h * t[:, None] * _S[None, :])
    inv = 1.0 / T - 1.0 / cfg.T_REF
    k1 = np.exp(np.clip(ln_k1 - E1_R * inv, -700, 700))
    k2 = np.exp(np.clip(ln_k2 - E2_R * inv, -700, 700))
    cum = lambda f: np.concatenate([np.zeros((f.shape[0], 1)),
                                    np.cumsum((f[:, 1:] + f[:, :-1]) * .5 * _dS, axis=1)], axis=1)
    K1, K2 = cum(k1) * t[:, None], cum(k2) * t[:, None]
    yA = np.exp(-K1)
    integ = t[:, None] * k1 * yA * np.exp(-(K2[:, -1][:, None] - K2))
    trapz = getattr(np, "trapezoid", None) or np.trapz     # NumPy 1.x / 2.x
    return np.clip(trapz(integ, dx=_dS, axis=1), 0.0, 1.0) * 100.0

_p5 = np.array([2.2, -6.5, 4.7e3, 7.6e4, 2.8])
_ref = simulate_analytic(_p5, D)
_etd = simulate(np.r_[_p5, 0.0, 0.0], D, cfg=CFG)
print(f"check 1  ETD vs quadrature (beta=0): mean|diff|={np.abs(_ref-_etd).mean():.5f} "
      f"max={np.abs(_ref-_etd).max():.4f} yield pts")

# --- check 2: axial convergence ---
_fine = simulate(np.r_[_p5, 0.0, 0.0], D, n_steps=1280, cfg=CFG)
print("check 2  axial convergence (vs 1280 steps):")
for _n in (40, 80, 160, 320):
    _c = simulate(np.r_[_p5, 0.0, 0.0], D, n_steps=_n, cfg=CFG)
    print(f"    n_steps={_n:5d}  mean|diff|={np.abs(_c-_fine).mean():.5f}  "
          f"max|diff|={np.abs(_c-_fine).max():.4f}")
print(f"\n  => n_steps = {CFG.n_steps} must be assessed against the errors printed above.")


## 4. Robust fitting

The implementation uses bounded nonlinear least squares with Cauchy loss, parameter scaling,
and multiple starting points. Cauchy loss reduces the influence of large residuals, but its
suitability remains an assumption to evaluate; this notebook does not infer a Student t noise
model or run MCMC.

Each fold uses its training portion to fit the physical parameters. Restarts explore different
initializations within the original parameter bounds; they do not guarantee a global optimum.
The fixed starting vector and bounds are retained from the original notebook. Their original
selection history is unavailable, so a strict audit cannot rule out prior full data influence.

The original beta bounds are [-40, 60]. Preserve them when comparing this version with the original
implementation and inspect boundary estimates before giving them a physical interpretation.


In [ ]:
print("Available physical ablations:", list(VARIANTS))
print("Loss:", CFG.robust_loss, "| full-data restarts:", CFG.n_restarts)


## 5. Parameter fits using all data
Fit the physical ablations and inspect their parameters and training residuals. These are
in sample diagnostics. The cross validation section below is the appropriate comparison for
predictive generalization.


In [ ]:
ladder = {}
for name in VARIANTS:
    t0 = time.perf_counter()
    p = fit(D, y, variant=name, cfg=CFG)
    pred = simulate(p, D, cfg=CFG)
    r = pred - y
    inl = np.abs(r) < 10
    ladder[name] = p
    print(f"{name:14s} RMSE={rmse(pred,y):7.3f}  trim={trimmed_rmse(pred,y):6.3f}  "
          f"MedAE={medae(pred,y):6.4f}  inliers={inl.sum():3d}/{len(y)}  "
          f"E2={p[3]*CFG.R_GAS/1000:7.1f} kJ/mol  ({time.perf_counter()-t0:.0f}s)")

P_STAR = ladder["two_enthalpy"]
phys_train = simulate(P_STAR, D, cfg=CFG)
res_train = phys_train - y
ln_k1, ln_k2, E1_R, E2_R, h, beta1, beta2 = P_STAR

print("\nFITTED EFFECTIVE PARAMETERS")
print(f"  k1(425K) = {np.exp(ln_k1):10.4f}      E1 = {E1_R*CFG.R_GAS/1000:7.1f} kJ/mol")
print(f"  k2(425K) = {np.exp(ln_k2):10.6f}      E2 = {E2_R*CFG.R_GAS/1000:7.1f} kJ/mol")
print(f"  h        = {h:10.4f}")
print(f"  beta1    = {beta1:10.4f} K.L/mol   ({beta1*C0.max():+.1f} K at max concentration)")
print(f"  beta2    = {beta2:10.4f} K.L/mol   ({beta2*C0.max():+.1f} K at max concentration)")

inl = np.abs(res_train) < 10
inlier_rmse = rmse(phys_train[inl], y[inl]) if inl.any() else float("nan")
correlation = (pearsonr(C0[inl], res_train[inl]).statistic
               if inl.sum() >= 2 and np.ptp(C0[inl]) > 0 and np.ptp(res_train[inl]) > 0
               else float("nan"))
print(f"MedAE={medae(phys_train,y):.4f}; inlier RMSE={inlier_rmse:.4f}; "
      f"residual concentration correlation={correlation:+.3f}")
print("NaN means a diagnostic is unavailable for the selected inlier subset.")

print("Interpret coefficient signs conditionally; they are effective estimates, not proof of the mechanism.")


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))

ax[0].scatter(y, phys_train, s=26, alpha=.7, color="#2b6cb0")
ax[0].plot([0, 100], [0, 100], "--", color="#c53030")
ax[0].set_xlabel("observed"); ax[0].set_ylabel("physics model")
ax[0].set_title("Parity (in-sample, robust fit)")

ax[1].hist(res_train, bins=60, color="#2b6cb0", edgecolor="white")
ax[1].set_xlim(-30, 30); ax[1].set_xlabel("residual")
ax[1].set_title(f"Sharp core, heavy tails (kurtosis {kurtosis(res_train):.1f})")

tt = np.linspace(0.05, 3.0, 120)
for T_probe, c in [(380, "#2b6cb0"), (420, "#38a169"), (460, "#dd6b20"), (500, "#c53030")]:
    d_probe = dict(tau=tt, Ti=np.full_like(tt, T_probe), Tj=np.full_like(tt, T_probe),
                   C0=np.full_like(tt, np.median(C0)), Q=np.full_like(tt, np.median(Q)),
                   L=np.full_like(tt, np.median(L)))
    ax[2].plot(tt, simulate(P_STAR, d_probe, cfg=CFG), color=c, lw=2, label=f"T = {T_probe} K")
ax[2].set_xlabel(r"$\tau=L/Q$"); ax[2].set_ylabel("yield of B (%)")
ax[2].set_title("Model-implied operating curves"); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

print("The right panel shows model-implied curves; external validation is needed before operational use.")


## 6. Repeated cross validation

Five folds over two random seeds. In every fold, fit each physical variant using training rows
only. The pure ML comparison uses an ExtraTrees hurdle model: a classifier estimates the chance
of nonzero yield, and a regressor estimates the yield conditional on that event.

Report raw RMSE, median absolute error, and trimmed RMSE (drop the largest 10% of squared errors).
Raw RMSE is the challenge metric. Trimmed RMSE is an auxiliary diagnostic that can hide errors
near difficult operating regimes; it does not measure error against unobserved clean labels.

The residual tree trains on in sample physical residuals within the training fold. Validation
labels are not used for fitting either component. The later λ sweep uses these same validation
predictions, so tuning and reporting are not fully separated.


In [ ]:
LAMS = [0.0, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0]
records, lam_store = [], {l: ([], []) for l in LAMS}

for seed in CFG.seeds:
    store = {k: ([], []) for k in list(VARIANTS) + ["pure_ML", "hybrid"]}
    for tr_i, va_i in KFold(CFG.n_splits, shuffle=True, random_state=seed).split(y):
        d_tr, d_va = sub(D, tr_i), sub(D, va_i)

        for name in VARIANTS:                      # the ablation, cross-validated
            p_f = fit(d_tr, y[tr_i], variant=name, n_restarts=8, seed=100 + seed, max_nfev=400, cfg=CFG)
            store[name][0].append(simulate(p_f, d_va, cfg=CFG)); store[name][1].append(y[va_i])
            if name == "two_enthalpy":
                ph_tr, ph_va = simulate(p_f, d_tr, cfg=CFG), simulate(p_f, d_va, cfg=CFG)

        X_tr, X_va = features(d_tr, ph_tr), features(d_va, ph_va)

        # pure ML baseline: soft hurdle. Under squared error the optimal prediction
        # is E[y|x] = P(live|x) * E[y|x,live], so we MULTIPLY rather than threshold.
        # NOTE: 8 features, not 20.
        store["pure_ML"][0].append(
            hurdle_predict(X_tr[:, :8], y[tr_i], X_va[:, :8], seed=seed))
        store["pure_ML"][1].append(y[va_i])

        corr = ExtraTreesRegressor(400, random_state=seed, n_jobs=-1) \
                 .fit(X_tr, y[tr_i] - ph_tr).predict(X_va)
        store["hybrid"][0].append(np.clip(ph_va + CFG.lambda_residual * corr, 0, 100))
        store["hybrid"][1].append(y[va_i])
        for l in LAMS:
            lam_store[l][0].append(np.clip(ph_va + l * corr, 0, 100))
            lam_store[l][1].append(y[va_i])

    for name, (pl, yl) in store.items():
        p_, y_ = np.concatenate(pl), np.concatenate(yl)
        records.append(dict(seed=seed, model=name, RMSE=rmse(p_, y_),
                            trimRMSE=trimmed_rmse(p_, y_), MedAE=medae(p_, y_)))
    print(f"  seed {seed} done", flush=True)

cv = pd.DataFrame(records).groupby("model")[["RMSE", "trimRMSE", "MedAE"]].mean().round(3)
display(cv.sort_values("trimRMSE"))

cv.to_csv(Path(CFG.out_dir) / "cv_summary.csv")
pd.DataFrame(records).to_csv(Path(CFG.out_dir) / "cv_by_seed.csv", index=False)


## 7. Residual weight sensitivity

λ = 0 gives pure physics; λ = 1 applies the full ML correction. Inspect whether raw and trimmed
metrics favor different weights. The original submitted setting of 0.30 is retained for continuity.
A difference between the two curves does not prove that the residual model is learning noise.

Selecting λ on these validation predictions introduces selection optimism. Nested CV or an
independent holdout is needed to estimate the performance of the selection procedure. The supplied
project includes no executed nested CV result.


In [ ]:
rows = []
for l in LAMS:
    p_, y_ = np.concatenate(lam_store[l][0]), np.concatenate(lam_store[l][1])
    rows.append({"lambda": l, "RMSE_raw": rmse(p_, y_),
                 "trimRMSE_trimmed": trimmed_rmse(p_, y_), "MedAE": medae(p_, y_)})
lam_df = pd.DataFrame(rows).round(4)
display(lam_df)

best_trim = float(lam_df.loc[lam_df.trimRMSE_trimmed.idxmin(), "lambda"])
best_raw = float(lam_df.loc[lam_df.RMSE_raw.idxmin(), "lambda"])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(lam_df["lambda"], lam_df.RMSE_raw, "o-", color="#a0aec0", label="RMSE (all labels)")
ax2 = ax.twinx()
ax2.plot(lam_df["lambda"], lam_df.trimRMSE_trimmed, "s-", color="#2b6cb0", label="trimmed RMSE")
ax2.axvline(CFG.lambda_residual, ls="--", color="#c53030")
ax.set_xlabel("λ (residual-correction weight)")
ax.set_ylabel("RMSE (raw)", color="#718096"); ax2.set_ylabel("trimmed RMSE", color="#2b6cb0")
ax.set_title("Residual correction: sensitivity to the evaluation metric")
plt.tight_layout(); plt.show()

print(f"lambda minimising trimmed RMSE (trimmed diagnostic) = {best_trim}")
print(f"lambda minimising raw RMSE     (raw labels)      = {best_raw}")
print(f"submitted lambda                                   = {CFG.lambda_residual}")

lam_df.to_csv(Path(CFG.out_dir) / "lambda_sweep.csv", index=False)
print("Use nested CV or a held-out set before reporting a selected weight as an unbiased optimum.")


## 8. Final model and submission

In [ ]:
residual_model = ExtraTreesRegressor(400, random_state=RNG, n_jobs=-1) \
                    .fit(features(D, phys_train), y - phys_train)

phys_test = simulate(P_STAR, D_te, cfg=CFG)
X_test = features(D_te, phys_test)
predictions = np.clip(phys_test + CFG.lambda_residual * residual_model.predict(X_test),
                      CFG.y_min, CFG.y_max)

submission = pd.DataFrame({"overall_yield": np.round(predictions, 4)})
assert len(submission) == 50, "must be exactly 50 rows"
assert list(submission.columns) == ["overall_yield"], "one column named overall_yield"
assert submission.overall_yield.notna().all() and np.isfinite(submission.overall_yield).all()
assert (submission.overall_yield >= 0).all() and (submission.overall_yield <= 100).all()

sub_path = Path(CFG.out_dir) / f"{CFG.team_name}.csv"
submission.to_csv(sub_path, index=False)

print(f"wrote {sub_path}  ({len(submission)} rows, 4 dp)")
print(f"  mean={predictions.mean():.2f}  range=[{predictions.min():.3f}, {predictions.max():.3f}]")
print(f"  near-zero predictions: {(predictions < 0.5).sum()}")
print(f"  mean |physics - final| = {np.abs(phys_test - predictions).mean():.3f} "
      "(the ML term is a nudge, not a driver)")
display(submission.head(10))

In [ ]:
import joblib

bundle = {
    "physical_parameters": {
        "ln_k1_ref": float(P_STAR[0]), "ln_k2_ref": float(P_STAR[1]),
        "E1_over_R": float(P_STAR[2]), "E2_over_R": float(P_STAR[3]),
        "h": float(P_STAR[4]), "beta1": float(P_STAR[5]), "beta2": float(P_STAR[6]),
        "T_ref": CFG.T_REF,
        "E1_kJ_per_mol": float(P_STAR[2] * CFG.R_GAS / 1000),
        "E2_kJ_per_mol": float(P_STAR[3] * CFG.R_GAS / 1000),
    },
    "residual_model": residual_model,
    "lambda_residual": CFG.lambda_residual,
    "config": asdict(CFG),
    "cv_summary": cv.to_dict(),
}
joblib.dump(bundle, Path(CFG.out_dir) / "reactor_surrogate.joblib")
with open(Path(CFG.out_dir) / "parameters.json", "w") as f:
    json.dump(bundle["physical_parameters"], f, indent=2)


# Verify the serialized artifact, not only the in-memory dictionary.
reloaded = joblib.load(Path(CFG.out_dir) / "reactor_surrogate.joblib")
check = predict_yield(test, reloaded)
assert np.allclose(check, predictions, atol=1e-8), "saved-model round-trip mismatch"
print("Saved-model round-trip passed on all test rows")
t0 = time.perf_counter()
_ = predict_yield(test, reloaded)
dt = time.perf_counter() - t0
print(f"Measured batch inference: {1000 * dt / len(test):.3f} ms/row on this machine")


## 9. Interpretation and next experiments

Use the generated results to discuss the physical ablations, held out error, and sensitivity to λ.
Avoid claims about an exact recovered mechanism, a guaranteed global optimum, or readiness for plant control.

### Limits
* The model assumes ideal plug flow and first order A → B → C reactions.
* Apparent kinetic and thermal coefficients can compensate for omitted processes or geometry.
* λ selection and score reporting share CV predictions.
* A low trimmed error can coexist with large errors in the most difficult operating conditions.
* The public implementation has no Bayesian posterior, calibrated predictive intervals, fit for axial
  dispersion, estimation of reaction order, or nested CV. Historical references to those experiments remain
  in the unchanged archive, with evidence status in `docs/RESULTS.md`.
* The original input data and execution logs are required to reproduce challenge claims.

### Next experiments
Restore the challenge inputs, retain executed outputs and fold predictions, quantify uncertainty
across splits, test solver accuracy near thermal transitions, and evaluate genuinely unseen
operating regimes.

### Outputs
`artifacts/Outliers.csv` contains the new submission with 50 rows. The originally supplied submission
is preserved in `submissions/Outliers.csv`; this workflow never overwrites it.
